# Base Ranking & Adaptive Re-Ranking Model

## The Problem

Since no ground-truth `fit` values exist, we treat this as:

A content-based ranking problem with adaptive relevance feedback.

The system must:

- Rank candidates by role fitness
- Allow re-ranking when a candidate is starred
- Improve ranking behavior dynamically

In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# data load
# KEYWORDS = "aspiring human resources seeking human resources"
CSV_PATH = "../data/potential-talents.csv"

df = pd.read_csv(CSV_PATH)
df.head()

,id,job_title,location,connection,fit
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85,NaN
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500+,NaN
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,NaN
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+,NaN
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+,NaN


### Baseline Scoring Design

Based on EDA findings, I chose three scoring components:

1. HR Relevance Score
2. Seniority Score
3. Intent Score

These reflect domain knowledge derived from exploratory analysis.

In [2]:
def normalize(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text
    
df["job_title_n"] = df["job_title"].apply(normalize)


HR_TERMS = {
    "human resources": 3,
    " hr ": 3,
    "generalist": 2,
    "specialist": 2,
    "coordinator": 2,
    "people": 1,
    "development": 1,
    "talent": 1
}

SENIOR_TERMS = {
    "senior": 2,
    "manager": 2,
    "director": 3,
    "vp": 3,
    "chro": 4,
    "specialist": 1,
    "coordinator": 1
}

INTENT_TERMS = {
    "aspiring": 1,
    "seeking": 1
}

def score_text(text, term_dict):
    score = 0
    for term, weight in term_dict.items():
        if term in text:
            score += weight
    return score

df["hr_score"] = df["job_title_n"].apply(lambda x: score_text(x, HR_TERMS))
df["seniority_score"] = df["job_title_n"].apply(lambda x: score_text(x, SENIOR_TERMS))
df["intent_score"] = df["job_title_n"].apply(lambda x: score_text(x, INTENT_TERMS))

df["semantic_score"] = (
    0.6 * df["hr_score"]
    + 0.35 * df["seniority_score"]
    + 0.05 * df["intent_score"]
)

ranked = df.sort_values("semantic_score", ascending=False)

print(ranked[["id","job_title","semantic_score"]].head(20).to_string(index=False))

 id                                                         job_title  semantic_score
  6                               Aspiring Human Resources Specialist            3.40
 24                               Aspiring Human Resources Specialist            3.40
 36                               Aspiring Human Resources Specialist            3.40
 49                               Aspiring Human Resources Specialist            3.40
 60                               Aspiring Human Resources Specialist            3.40
 56  Human Resources Coordinator at InterContinental Buckhead Atlanta            3.35
 43  Human Resources Coordinator at InterContinental Buckhead Atlanta            3.35
 13  Human Resources Coordinator at InterContinental Buckhead Atlanta            3.35
 68                           Human Resources Specialist at Luxottica            3.35
 65  Human Resources Coordinator at InterContinental Buckhead Atlanta            3.35
 37 Student at Humber College and Aspiring Human Resou

#### Baseline Observations

- HR-related candidates rank above non-HR roles.
- Senior roles appear but do not dominate.
- Duplicate titles cluster near the top.

The baseline is stable but lacks diversity and adaptive capability.

In [3]:
def parse_connections(x):
    if isinstance(x, str) and "500+" in x:
        return 500
    try:
        return int(x)
    except:
        return 0
    
df["connections_num"] = df["connection"].apply(parse_connections)

# Min-Max normalize
df["connections_norm"] = (
    (df["connections_num"] - df["connections_num"].min()) /
    (df["connections_num"].max() - df["connections_num"].min())
)

role_location = "Houston"
if role_location:
    df["location_match"] = df["location"].str.contains(
        role_location, case=False, na=False
    ).astype(int)
else:
    df["location_match"] = 0

In [4]:
SEMANTIC_WEIGHT = 0.90
CONNECTION_WEIGHT = 0.07
LOCATION_WEIGHT = 0.03

df["base_score_final"] = (
    SEMANTIC_WEIGHT * df["semantic_score"] +
    CONNECTION_WEIGHT * df["connections_norm"] +
    LOCATION_WEIGHT * df["location_match"]
)

# Normalize again to keep scale consistent
df["base_score_final"] = (
    (df["base_score_final"] - df["base_score_final"].min()) /
    (df["base_score_final"].max() - df["base_score_final"].min())
)

ranked = df.sort_values("base_score_final", ascending=False)
print(ranked[["id","job_title","base_score_final"]].head(20).to_string(index=False))

 id                                                         job_title  base_score_final
 13  Human Resources Coordinator at InterContinental Buckhead Atlanta          1.000000
 65  Human Resources Coordinator at InterContinental Buckhead Atlanta          1.000000
 56  Human Resources Coordinator at InterContinental Buckhead Atlanta          1.000000
 43  Human Resources Coordinator at InterContinental Buckhead Atlanta          1.000000
 68                           Human Resources Specialist at Luxottica          1.000000
  6                               Aspiring Human Resources Specialist          0.991896
 24                               Aspiring Human Resources Specialist          0.991896
 36                               Aspiring Human Resources Specialist          0.991896
 60                               Aspiring Human Resources Specialist          0.991896
 49                               Aspiring Human Resources Specialist          0.991896
 53             Seeking Human Re

### Duplicate Title Penalty

To prevent repeated identical titles from dominating the top results, I introduce a small duplicate penalty:

This improves diversity while preserving score ordering.

In [5]:
# Duplicate penalty
title_counts = df["job_title_n"].value_counts()
df["dup_penalty"] = df["job_title_n"].map(title_counts)

df["base_score_adj"] = df["base_score_final"] - 0.05 * (df["dup_penalty"] - 1)

# Normalize adjusted baseline (stable baseline for ranking / cutoff)
df["base_score_adj_norm"] = (df["base_score_adj"] - df["base_score_adj"].min()) / (
    df["base_score_adj"].max() - df["base_score_adj"].min()
)

### Exploratory: Adaptive Weight Shifting

I initially attempted to adjust global scoring weights based on the starred candidate profile.
* Result: Minimal ranking changes.
* Reason: Feature space was highly correlated, limiting differentiation.
* Conclusion: Weight shifting alone is insufficient.

In [6]:
# base weights
weights = np.array([0.6, 0.35, 0.05])  # HR, Seniority, Intent

def rerank_with_star(df, starred_id, weights, lr=0.1):

    star = df[df["id"] == starred_id].iloc[0]

    star_profile = np.array([
        star["hr_score"],
        star["seniority_score"],
        star["intent_score"]
    ])

    # normalize star profile
    if star_profile.sum() > 0:
        star_profile = star_profile / star_profile.sum()

    # update weights
    new_weights = weights + lr * star_profile

    # normalize weights
    new_weights = new_weights / new_weights.sum()

    # compute new score
    df["adaptive_score"] = (
        new_weights[0] * df["hr_score"] +
        new_weights[1] * df["seniority_score"] +
        new_weights[2] * df["intent_score"]
    )
    df["adaptive_base_score_final"] = (
        SEMANTIC_WEIGHT * df["adaptive_score"] +
        CONNECTION_WEIGHT * df["connections_norm"] +
        LOCATION_WEIGHT * df["location_match"]
    )

    # Normalize again to keep scale consistent
    df["adaptive_base_score_final"] = (
        (df["adaptive_base_score_final"] - df["adaptive_base_score_final"].min()) /
        (df["adaptive_base_score_final"].max() - df["adaptive_base_score_final"].min())
    )

    ranked = df.sort_values("adaptive_base_score_final", ascending=False)

    return ranked, new_weights

reranked, new_weights = rerank_with_star(df, starred_id=89, weights=weights, lr=0.1) # 3 (HR), 61 (Senior HR), 89 (Director HR)
print(reranked[["id","job_title","adaptive_base_score_final"]].head(20).to_string(index=False))

 id                                                         job_title  adaptive_base_score_final
 13  Human Resources Coordinator at InterContinental Buckhead Atlanta                   1.000000
 65  Human Resources Coordinator at InterContinental Buckhead Atlanta                   1.000000
 56  Human Resources Coordinator at InterContinental Buckhead Atlanta                   1.000000
 43  Human Resources Coordinator at InterContinental Buckhead Atlanta                   1.000000
 68                           Human Resources Specialist at Luxottica                   1.000000
  6                               Aspiring Human Resources Specialist                   0.990481
 24                               Aspiring Human Resources Specialist                   0.990481
 36                               Aspiring Human Resources Specialist                   0.990481
 60                               Aspiring Human Resources Specialist                   0.990481
 49                           

## Similarity-Based Adaptive Re-Ranking

We transition to a content-based relevance feedback model.

Steps:

1. Represent job titles using TF-IDF vectors.
2. Compute cosine similarity between starred candidate and all others.
3. Fuse similarity with base score.

Adaptive formula:

adaptive_score = 0.6 * normalized_base_score + 0.4 * similarity_score

In [7]:
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(df["job_title_n"])

def rerank_with_similarity(df_in, starred_id, alpha=0.4):
    df = df_in.copy()

    star_index = df.index[df["id"] == starred_id][0]
    star_vector = tfidf_matrix[star_index]
    sim = cosine_similarity(star_vector, tfidf_matrix).flatten()

    df["similarity_to_star"] = sim
    df["adaptive_score"] = (1 - alpha) * df["base_score_adj_norm"] + alpha * df["similarity_to_star"]

    return df.sort_values("adaptive_score", ascending=False)

reranked_tfidf = rerank_with_similarity(df, starred_id=89, alpha=0.8) # 3 (HR), 61 (Senior HR), 89 (Director HR)
print(reranked_tfidf[["id","job_title","adaptive_score"]].head(20).to_string(index=False))

 id                                                                                                             job_title  adaptive_score
 89                                                                                       Director Human Resources  at EY        0.974054
 69                                                            Director of Human Resources North America, Groupe Beneteau        0.424390
104                                                                      Director Of Administration at Excellence Logging        0.306406
 73                                              Aspiring Human Resources Manager, seeking internship in Human Resources.        0.297906
  6                                                                                   Aspiring Human Resources Specialist        0.290524
 24                                                                                   Aspiring Human Resources Specialist        0.290524
 36                               

#### Why Similarity-Based Re-Ranking Works

Unlike global weight shifting, prototype similarity:

- Captures nuanced textual similarity
- Scales with small datasets
- Is interpretable
- Mimics search relevance feedback

This aligns with real-world recruiter behavior.

### Scenario Testing

We test three starring scenarios:

1. Star entry-level aspirant
2. Star senior specialist
3. Star director-level HR

Observation:

- Starring an aspirant promotes similar aspirant profiles.
- Starring a senior promotes other senior profiles.
- Starring a director promotes executive-level candidates.

This confirms adaptive behavior.

In [8]:
def rank_position(df, candidate_id):
    df_reset = df.reset_index(drop=True)
    return df_reset.index[df_reset["id"] == candidate_id][0] + 1


def compute_rank_changes(before_df, after_df):
    before_df = before_df.reset_index(drop=True)
    after_df = after_df.reset_index(drop=True)

    before_df["rank_before"] = before_df.index + 1
    after_df["rank_after"] = after_df.index + 1

    merged = before_df[["id", "rank_before"]].merge(
        after_df[["id", "rank_after"]],
        on="id"
    )

    merged["rank_change"] = merged["rank_before"] - merged["rank_after"]

    return merged

In [ ]:
star_id = 89  # example director id

# BEFORE: baseline ranking
baseline_ranked = df.sort_values("base_score_adj_norm", ascending=False)
# compute similarity once for baseline too
star_index = df.index[df["id"] == star_id][0]
sim = cosine_similarity(tfidf_matrix[star_index], tfidf_matrix).flatten()
baseline_ranked["similarity_to_star"] = sim

# AFTER: adaptive ranking
adaptive_ranked = rerank_with_similarity(df.copy(), star_id)

# Rank of starred candidate
rank_before = rank_position(baseline_ranked, star_id)
rank_after = rank_position(adaptive_ranked, star_id)

print("Starred Candidate Rank:")
print("Before:", rank_before)
print("After :", rank_after)

top10_before = baseline_ranked.head(10)["similarity_to_star"].mean()
top10_after = adaptive_ranked.head(10)["similarity_to_star"].mean()

print("\nMean Similarity of Top 10:")
print("Before:", round(top10_before, 4))
print("After :", round(top10_after, 4))

rank_changes = compute_rank_changes(baseline_ranked, adaptive_ranked)

# Define similar candidates (top 25% similarity)
top_similar_ids = (
    baseline_ranked.sort_values("similarity_to_star", ascending=False)
      .iloc[1:11]["id"]  # skip starred itself
)

similar_ids = top_similar_ids
dissimilar_ids = df[~df["id"].isin(top_similar_ids)]["id"]

mean_similar_movement = rank_changes[rank_changes["id"].isin(similar_ids)]["rank_change"].mean()
mean_dissimilar_movement = rank_changes[rank_changes["id"].isin(dissimilar_ids)]["rank_change"].mean()

print("\nAverage Rank Movement:")
print("Similar Candidates   :", round(mean_similar_movement, 2))
print("Dissimilar Candidates:", round(mean_dissimilar_movement, 2))


Total Candidates: 104
Remaining After Soft Filter: 50

Starred Candidate Rank:
Before: 10
After : 1

Mean Similarity of Top 10:
Before: 0.0681
After : 0.2087

Average Rank Movement:
Similar Candidates   : -0.4
Dissimilar Candidates: 0.04


#### Result

| Scenario (id) | Star Rank Before | Star Rank After | Mean Top10 Similarity Before | Mean After | Similar Candidates Movement| Disimilar Candidates Movement |
| -------- | ---------------- | --------------- | ---------------------------- | ---------- | ----- | ------ |
| Starred Director HR (89) | 19 | 1 | 0.1233 | 0.2375 | 8.7 | -0.93 |
| Starred Senior HR (61) | 45 | 5 | 0.1233 | 0.6823 | -1.6 | 0.17 |
| Starred Aspiring HR Professional (3) | 70 | 8 | 0.1233 | 0.7621 | 41.7 | -4.44 |
| Starred HR Coordinator (43) | 3 | 4 | 0.1233 | 0.4769 | 5.6 | -0.6 |
| Non HR but profile mentions HR (70) | 31 | 1 | 0.1233 | 0.1572 | 5.7 | -0.61 |
| Non HR (80) | 87 | 40 | 0.1233 | 0.0 | -0.7 | 0.7 |

#### Note on Rank Redistribution

Because ranking is relative, large upward movement of highly similar candidates may
cause moderate similar candidates to shift slightly downward in average movement statistics.

Therefore, we prioritize:

- Star candidate rank improvement
- Increase in mean top-k similarity

as the primary indicators of adaptive alignment.

## Soft Filtering Strategy

To remove clearly irrelevant candidates without limiting adaptability,
we apply filtering *after adaptive ranking*.

A candidate is excluded only if:

- Their normalized base score falls below a dynamic percentile cutoff, AND
- Their similarity to the starred candidate is below a similarity threshold.

This preserves borderline candidates while removing clearly irrelevant ones.

In [17]:
# After rerank_with_similarity(...)
ranked_for_filter = adaptive_ranked.copy()

percentile_cutoff = 0.70
cutoff_value = ranked_for_filter["base_score_adj_norm"].quantile(percentile_cutoff)

similarity_threshold = 0.10

ranked_for_filter["is_filtered_out"] = (
    (ranked_for_filter["base_score_adj_norm"] < cutoff_value) &
    (ranked_for_filter["similarity_to_star"] < similarity_threshold)
)

filtered_ranked = ranked_for_filter[~ranked_for_filter["is_filtered_out"]].copy()

print("Total Candidates:", len(ranked_for_filter))
print("Remaining After Soft Filter:", len(filtered_ranked))

Total Candidates: 104
Remaining After Soft Filter: 50


#### Why Percentile-Based Cutoff?

Absolute score thresholds (e.g., score > 0.6) do not generalize across roles
because score distributions vary depending on keyword density and domain overlap.

Using percentile-based filtering ensures:

- Distribution awareness
- Role-agnostic behavior
- Stability across different search queries
- Avoidance of arbitrary score assumptions

#### Bias Mitigation Considerations

To reduce potential bias in candidate ranking, we incorporate the following design decisions:

1. Minimal Use of Sensitive Signals
- Location is not heavily weighted.
- Connection count is not used as a primary ranking feature.

2. Soft Filtering Instead of Hard Exclusion
Hard filtering can introduce structural bias.
Soft filtering allows adaptive recovery through semantic similarity.

3. Recruiter-Driven Adaptation
Rather than encoding static preferences (e.g., senior > junior),
the system adapts based on starring behavior.

This shifts control from system designer to recruiter signal.

4. Duplicate Diversity Control
Duplicate job titles are slightly penalized to prevent monoculture ranking dominance.

5. Future Enhancements
- Blind ranking mode (hide location during scoring)
- Multi-star consensus weighting
- Embedding-based semantic similarity to reduce lexical bias
- Fairness auditing if demographic features become available